# 07 - Model fraud v1.1: tanpa atribut terproteksi, siap impor
Model juara v1 (`XGB mask`, notebook 02b-03) masih memakai **gender, usia, dan wilayah** di antara 68 fiturnya, padahal proposal §6.4 menyatakan *'no protected attribute in the model'*.
Notebook 06 menunjukkan bahwa menghapusnya tidak merugikan. Di sini keputusan itu dijalankan menjadi model baru, lengkap dengan bundel export.

**Dua koreksi dalam v1.1**
1. **Atribut terproteksi dihapus:** `gender`, `age_ord`, `region`.
2. **Fitur 'tenure' yang cacat dihapus:** `tenure_months` dan `hist_days` di training dihitung sejak 2025-07-01 (awal jendela data), bukan sejak rekening dibuka. Di produksi nilainya akan berbeda dari yang dilihat model. Daripada mereproduksi cacat itu, kedua fitur dikeluarkan.

**Alur:** (1) bandingkan tiga set fitur pada DEV dengan 3 seed; (2) latih model akhir; (3) tetapkan ambang pita dari DEV-OOT; (4) ekspor bundel portabel dan uji paritas di proses bersih. Evaluasi pada data baru (TEST2) ada di `07b_final_test_v1_1.ipynb`.
Data TEST2 tidak disentuh di notebook ini.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")      # notebooks use paths relative to the project root
print("working directory:", os.getcwd())

In [1]:
import os, sys, json, time, shutil, subprocess, warnings, joblib
os.environ.setdefault("OMP_NUM_THREADS", "10"); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd()); import splits as S
from sklearn.metrics import average_precision_score, precision_recall_curve
from xgboost import XGBClassifier
import xgboost as xgb

dev = S.load_dev()
for c in S.CAT: dev[c] = dev[c].astype("category")
pool, oot = S.cv_pool(dev), S.dev_oot(dev); L, W = S.LABEL, S.WEIGHT
FEATS = S.features(dev); PROT = ["gender", "age_ord", "region"]; QUIRK = ["tenure_months", "hist_days"]
P = dict(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, min_child_weight=5.0, reg_lambda=1.0, gamma=0.0)   # bawaan, sama seperti v1
BUNDLE, ART = "bundle/fraud_xgb_mask_v1_1", "artifacts"
print(f"pool {len(pool):,} | DEV-OOT {len(oot):,} | fitur v1: {len(FEATS)}")

pool 359,008 | DEV-OOT 129,799 | fitur v1: 68


## 1. Tiga set fitur, tiga seed
Resep pelatihan sama dengan v1 (`mask`: data asli + satu salinan yang 0-40% selnya dikosongkan), hyperparameter bawaan. Evaluasi pada DEV Jul-Sep: PR-AUC berbobot saat data lengkap dan saat 20% nilai hilang.

**Aturan pilih (ditetapkan sebelum melihat hasil):** pilih set paling kecil yang tidak memuat atribut terproteksi, asalkan PR-AUC bersihnya tidak lebih rendah dari set penuh melebihi satu simpangan seed.

In [2]:
def augment(df, cols, seed):
    r = np.random.default_rng(seed); d = df.copy(); rate = r.uniform(0, 0.4, len(d))
    for c in cols: d[c] = d[c].mask(r.random(len(d)) < rate)
    return pd.concat([df, d], ignore_index=True)
def fit(cols, seed=42):
    t = augment(pool[cols + [L]], cols, 2000 + seed)
    return XGBClassifier(tree_method="hist", enable_categorical=True, n_jobs=10, random_state=seed, **P).fit(t[cols], t[L])
def pr(m, cols, rate=0.0, d=None):
    d = oot if d is None else d; X = d[cols].copy()
    if rate:
        r = np.random.default_rng(7)
        for c in cols: X[c] = X[c].mask(r.random(len(X)) < rate)
    return average_precision_score(d[L], m.predict_proba(X)[:, 1], sample_weight=d[W])
sets = {"A: 68 fitur (model v1)": FEATS, "B: tanpa terproteksi (65)": [c for c in FEATS if c not in PROT], "C: tanpa terproteksi dan tenure/hist (63)": [c for c in FEATS if c not in PROT + QUIRK]}
rows, t0 = {}, time.time()
for name, cols in sets.items():
    ms = [fit(cols, s) for s in (42, 1, 2)]; cl = [pr(m, cols) for m in ms]; mi = [pr(m, cols, 0.2) for m in ms]
    rows[name] = {"PR-AUC bersih (rata-rata 3 seed)": np.mean(cl), "std seed": np.std(cl), "20% hilang (rata-rata)": np.mean(mi), "std seed (20% hilang)": np.std(mi), "jumlah fitur": len(cols)}
    print(name, f"{time.time() - t0:.0f}s", flush=True)
audit = pd.DataFrame(rows).T; audit.round(4)

A: 68 fitur (model v1) 73s


B: tanpa terproteksi (65) 183s


C: tanpa terproteksi dan tenure/hist (63) 283s


,PR-AUC bersih (rata-rata 3 seed),std seed,20% hilang (rata-rata),std seed (20% hilang),jumlah fitur
A: 68 fitur (model v1),0.6607,0.0146,0.4856,0.0216,68.0
B: tanpa terproteksi (65),0.6619,0.0109,0.4717,0.0071,65.0
C: tanpa terproteksi dan tenure/hist (63),0.6696,0.0075,0.5109,0.0023,63.0


In [3]:
base = audit.iloc[0]
eligible = [n for n in audit.index if not any(p in sets[n] for p in PROT) and audit.loc[n, "PR-AUC bersih (rata-rata 3 seed)"] >= base["PR-AUC bersih (rata-rata 3 seed)"] - base["std seed"]]
chosen = min(eligible, key=lambda n: len(sets[n])); F11 = sets[chosen]
print("memenuhi syarat:", eligible); print("DIPILIH:", chosen, "->", len(F11), "fitur")
assert not any(p in F11 for p in PROT + QUIRK)

memenuhi syarat: ['B: tanpa terproteksi (65)', 'C: tanpa terproteksi dan tenure/hist (63)']
DIPILIH: C: tanpa terproteksi dan tenure/hist (63) -> 63 fitur


**Cara membaca:** set C tidak hanya memenuhi syarat, tetapi hasilnya sama atau lebih baik dan lebih stabil antar seed. Artinya gender, usia, wilayah, dan tenure-cacat tidak membawa sinyal yang dibutuhkan model; yang tersisa adalah sinyal perilaku (payee, jam malam, perangkat, pass-through, merchant) dan profil non-terproteksi (pendapatan, pekerjaan, segmen, umur KYC).

## 2. Model akhir v1.1 dan ambang pita
Seed 42, dilatih pada seluruh pool. Ambang pita dihitung dari DEV-OOT (bukan TEST2): *suspicious* = presisi >= 5%, *high-risk* = presisi >= 50%, setelah koreksi prior untuk negatif yang di-down-sample 3%.

In [4]:
model = fit(F11, 42); p_oot = model.predict_proba(oot[F11])[:, 1]
print(f"DEV-OOT PR-AUC bersih {average_precision_score(oot[L], p_oot, sample_weight=oot[W]):.4f} | 20% hilang {pr(model, F11, 0.2):.4f}")
def calibrate(p, k=S.NEG_KEEP): return p * k / (p * k + (1 - p))
def bands(y, s, w, prec_sus=0.05, prec_high=0.5):
    p_, r_, th = precision_recall_curve(y, s, sample_weight=w); first = lambda t: th[np.argmax(p_[:-1] >= t)] if (p_[:-1] >= t).any() else th[-1]
    return float(min(first(prec_sus), first(prec_high))), float(first(prec_high))
t_s, t_h = bands(oot[L].values, calibrate(p_oot), oot[W].values); print(f"ambang: suspicious {t_s:.5f} | high_risk {t_h:.5f}")

DEV-OOT PR-AUC bersih 0.6590 | 20% hilang 0.5077
ambang: suspicious 0.00247 | high_risk 0.04664


## 3. Bundel export v1.1
Struktur sama dengan v1 (`model.json`, `schema.json`, `score.py`, `requirements.txt`, `MODEL_CARD.md`, `parity_report.json`). `schema.json` kini mencatat `protected_attributes_in_model: []` sebagai pernyataan yang dapat diperiksa mesin.

In [5]:
booster = model.get_booster(); assert list(booster.feature_names) == F11
GROUPS = {"device": [c for c in F11 if c.startswith(("new_device", "cred_change", "device_cnt"))], "geo": [c for c in F11 if c.startswith("geo_mismatch")], "merchant": [c for c in F11 if c.startswith("mix_")],
          "payee": [c for c in F11 if c.startswith(("new_payee", "distinct", "payee_concentration", "top_payee", "ftp_large"))], "malam": [c for c in F11 if c.startswith("night_share")],
          "saldo": [c for c in F11 if c.startswith(("passthrough", "out_to_in", "out_to_balance", "min_balance", "balance", "days_low"))],
          "profil": [c for c in ("declared_income", "shared_device_flag", "occupation", "segment", "kyc_age_months") if c in F11]}
assert all(GROUPS.values()), "setiap grup harus punya anggota"
os.makedirs(BUNDLE, exist_ok=True); booster.save_model(f"{BUNDLE}/model.json")
CATS = [c for c in F11 if c in S.CAT]
schema = dict(version="1.1", model="XGB|mask", created=time.strftime("%Y-%m-%d"), neg_keep=S.NEG_KEEP, alert_rate=0.001, protected_attributes_in_model=[], excluded_for_quirk=QUIRK,
              thresholds=dict(suspicious=t_s, high_risk=t_h, note="berlaku pada 'score' (terkoreksi prior); suspicious = presisi >= 5% di DEV-OOT, high_risk = presisi >= 50%"),
              null_policy="fitur kosong/tidak ada = tidak tersedia; kategori tidak dikenal = tidak tersedia",
              features=[dict(name=f, type="categorical" if f in CATS else "numeric", **({"categories": [str(x) for x in dev[f].cat.categories]} if f in CATS else {})) for f in F11], groups=GROUPS)
json.dump(schema, open(f"{BUNDLE}/schema.json", "w"), indent=1, ensure_ascii=False)
shutil.copy("deploy/score.py", f"{BUNDLE}/score.py")
open(f"{BUNDLE}/requirements.txt", "w").write(f"xgboost=={xgb.__version__}\nnumpy=={np.__version__}\npandas=={pd.__version__}\n")
print(sorted(os.listdir(BUNDLE)), f"| model.json {os.path.getsize(f'{BUNDLE}/model.json') / 1e3:.0f} KB")

['model.json', 'requirements.txt', 'schema.json', 'score.py'] | model.json 1002 KB


## 4. Uji paritas di proses bersih
Sama dengan 02c: 5.000 baris DEV-OOT, versi 20% sel kosong, dan baris dengan kategori tak dikenal. Skor bundel (proses `python -I` yang hanya memuat bundel) harus identik dengan model asli, dan latensi 200 request tunggal dicatat.

In [6]:
rng = np.random.default_rng(0); base_ = oot.sample(5000, random_state=1)[F11].reset_index(drop=True); miss = base_.copy()
for c in F11: miss[c] = miss[c].mask(rng.random(len(miss)) < 0.2)
unk = base_.head(500).copy()
for c in CATS: unk[c] = unk[c].astype(object); unk.loc[unk.index[:250], c] = "KATEGORI_BARU"
test_in = pd.concat([base_.assign(kasus="bersih"), miss.assign(kasus="miss20"), unk.assign(kasus="kategori_baru")], ignore_index=True)
for c in CATS: test_in[c] = test_in[c].astype(object)
inp = f"{BUNDLE}/_parity_input.parquet"; test_in.to_parquet(inp, index=False)
ref_df = test_in[F11].copy()
for c in CATS:
    cats = next(f["categories"] for f in schema["features"] if f["name"] == c)
    ref_df[c] = pd.Categorical(ref_df[c].where(ref_df[c].isin(cats)), categories=dev[c].cat.categories)
ref_raw = model.predict_proba(ref_df)[:, 1]
tmp_out = os.path.join(BUNDLE, "_parity_out.npz")
r = subprocess.run([sys.executable, "-I", "deploy/parity_check.py", BUNDLE, inp, tmp_out], capture_output=True, text=True); print(r.stdout.strip(), r.stderr.strip()[-300:])
out = np.load(tmp_out); diff = np.abs(out["raw"] - ref_raw)
by_case = pd.DataFrame({"kasus": test_in.kasus, "selisih": diff}).groupby("kasus").selisih.agg(["max", "mean"]); display(by_case)
assert diff.max() < 1e-6, "paritas gagal"
json.dump(dict(max_abs_diff=float(diff.max()), cases=by_case.round(9).to_dict(), latency_single_request=json.loads(r.stdout.strip().splitlines()[-1]), n_rows=int(len(test_in))), open(f"{BUNDLE}/parity_report.json", "w"), indent=1)
os.remove(inp); os.remove(tmp_out)
joblib.dump(model, "models/fraud_v1_1_xgb_mask.joblib")

{"n_requests": 200, "p50_ms": 35.5, "p95_ms": 44.7, "max_ms": 72.0} 


,max,mean
kasus,,
bersih,0.0,0.0
kategori_baru,0.0,0.0
miss20,0.0,0.0


['models/fraud_v1_1_xgb_mask.joblib']

In [7]:
card = f'''# Model card - fraud risk score v1.1 (XGB `mask`, tanpa atribut terproteksi)
**Status:** kandidat juara untuk data **sintetis** (SAS Hackathon 2026). Menggantikan v1; evaluasi akhir pada data baru ada di `07b_final_test_v1_1.ipynb`.

## Perubahan dari v1
* Gender, usia, dan wilayah **tidak** menjadi fitur (proposal §6.4); `schema.json` mencatat `protected_attributes_in_model: []`.
* `tenure_months` dan `hist_days` dikeluarkan karena dihitung dari awal jendela data, bukan dari pembukaan rekening.
* {len(F11)} fitur (v1: 68). Resep pelatihan dan hyperparameter sama (augmentasi `mask`, XGBoost bawaan).

## Hasil DEV Jul-Sep (rata-rata 3 seed)
PR-AUC bersih {audit.loc[chosen, "PR-AUC bersih (rata-rata 3 seed)"]:.3f} (std seed {audit.loc[chosen, "std seed"]:.3f}); 20% nilai hilang {audit.loc[chosen, "20% hilang (rata-rata)"]:.3f} (std {audit.loc[chosen, "std seed (20% hilang)"]:.3f}). Model v1: {base["PR-AUC bersih (rata-rata 3 seed)"]:.3f} dan {base["20% hilang (rata-rata)"]:.3f}.

## Ambang pita (pada `score`)
suspicious >= {t_s:.5f} (presisi >= 5% di DEV-OOT); high_risk >= {t_h:.5f} (presisi >= 50%).

## Keterbatasan
Data sintetis; angka memvalidasi pipeline, bukan perilaku produksi. Fraud stealth tetap lebih sulit. Model dilatih pada agregat akhir hari. Seleksi, ambang, dan klaim performa harus diulang pada data bank sebelum dipakai.
'''
open(f"{BUNDLE}/MODEL_CARD.md", "w").write(card)
json.dump(dict(features=F11, protected=PROT, quirk=QUIRK, audit=audit.round(5).to_dict(), chosen=chosen, thresholds=dict(suspicious=t_s, high_risk=t_h)), open(f"{ART}/model_v1_1.json", "w"), indent=1)
sorted(os.listdir(BUNDLE))

['MODEL_CARD.md',
 '__pycache__',
 'model.json',
 'parity_report.json',
 'requirements.txt',
 'schema.json',
 'score.py']